# Olist Brazilian E-Commerce: Sales, Delivery and Customer Analysis

Exploratory data analysis of about 100K orders (Sep 2016 to Oct 2018) from Olist, a Brazilian online marketplace.

**Business questions**
1. How did revenue and orders grow over time?
2. Which product categories and states drive revenue?
3. How good is delivery performance, and does it affect customer reviews?
4. How do customers pay, and how many come back?
5. How concentrated is revenue among sellers?

**Data:** Kaggle "Brazilian E-Commerce Public Dataset by Olist" (8 tables used, geolocation not needed).
Revenue = sum of item `price` on delivered orders (excludes freight). Currency: Brazilian real (R$).

## 1. Setup and load data
In Colab, run the next cell and select all the Olist CSV files (except geolocation) when asked.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
money = FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M' if x >= 1e6 else f'R${x/1e3:.0f}K')

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
dates = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
         'order_delivered_customer_date', 'order_estimated_delivery_date']
orders    = pd.read_csv('olist_orders_dataset.csv', parse_dates=dates)
items     = pd.read_csv('olist_order_items_dataset.csv')
payments  = pd.read_csv('olist_order_payments_dataset.csv')
reviews   = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
products  = pd.read_csv('olist_products_dataset.csv')
sellers   = pd.read_csv('olist_sellers_dataset.csv')
trans     = pd.read_csv('product_category_name_translation.csv')

for name, df in [('orders', orders), ('items', items), ('payments', payments), ('reviews', reviews),
                 ('customers', customers), ('products', products), ('sellers', sellers)]:
    print(f'{name:10s} {df.shape}')

## 2. Data quality checks

In [ ]:
print(orders['order_status'].value_counts())
print()
print('Purchase date range:', orders['order_purchase_timestamp'].min().date(), 'to', orders['order_purchase_timestamp'].max().date())
print('Duplicate order_ids:', orders['order_id'].duplicated().sum())
print('Orders with more than one review row:', reviews['order_id'].duplicated().sum())
print('Products with missing category:', products['product_category_name'].isna().sum())
print()
print(orders.isna().sum()[lambda s: s > 0])

**Decisions**
- Only `delivered` orders (97% of all orders) are used for revenue and delivery analysis.
- Reviews: some orders have several review rows, so I average the score per order.
- Delivery metrics skip the few delivered orders that have no delivery date.
- Category names are translated to English; missing categories are labelled `unknown`.

## 3. Build the analysis tables

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['month'] = delivered['order_purchase_timestamp'].dt.to_period('M')

order_rev = items.groupby('order_id')['price'].sum().rename('order_value')
review_score = reviews.groupby('order_id')['review_score'].mean().rename('review_score')

df = (delivered.merge(order_rev, on='order_id', how='left')
               .merge(review_score, on='order_id', how='left')
               .merge(customers, on='customer_id', how='left'))

# delivery metrics (only orders with a delivery date)
dl = df.dropna(subset=['order_delivered_customer_date']).copy()
dl['delivery_days'] = (dl['order_delivered_customer_date'] - dl['order_purchase_timestamp']).dt.total_seconds() / 86400
dl['late'] = dl['order_delivered_customer_date'] > dl['order_estimated_delivery_date']

# item level with category
it = (items.merge(delivered[['order_id', 'month']], on='order_id')
           .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
           .merge(trans, on='product_category_name', how='left'))
it['category'] = it['product_category_name_english'].fillna(it['product_category_name']).fillna('unknown')
print(df.shape, dl.shape, it.shape)

## 4. Headline KPIs

In [ ]:
total_rev = it['price'].sum()
n_orders = delivered['order_id'].nunique()
aov = df['order_value'].mean()
orders_per_cust = df.groupby('customer_unique_id')['order_id'].nunique()
repeat_rate = (orders_per_cust > 1).mean()

kpis = pd.DataFrame({
    'KPI': ['Total revenue (delivered, excl. freight)', 'Delivered orders', 'Unique customers',
            'Average order value', 'Repeat customer rate', 'Avg delivery time (days)',
            'Late delivery rate', 'Avg review score'],
    'Value': [f'R$ {total_rev:,.0f}', f'{n_orders:,}', f'{orders_per_cust.shape[0]:,}',
              f'R$ {aov:,.2f}', f'{repeat_rate:.1%}', f'{dl["delivery_days"].mean():.1f}',
              f'{dl["late"].mean():.1%}', f'{df["review_score"].mean():.2f}']
})
print(kpis.to_string(index=False))
kpis.to_csv('kpis.csv', index=False)

## 5. Revenue and orders over time
Only full months (Jan 2017 to Aug 2018) are shown. The first and last months of the dataset are partial.

In [ ]:
monthly = it.groupby('month').agg(revenue=('price', 'sum'), orders=('order_id', 'nunique'))
monthly = monthly.loc['2017-01':'2018-08']
peak = monthly['revenue'].idxmax()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(monthly.index.astype(str), monthly['revenue'], marker='o', color='#1f6feb')
ax[0].yaxis.set_major_formatter(money)
ax[0].set_title('Monthly revenue')
ax[1].bar(monthly.index.astype(str), monthly['orders'], color='#2da44e')
ax[1].set_title('Monthly delivered orders')
for a in ax:
    a.tick_params(axis='x', rotation=70)
    a.set_xlabel('')
plt.tight_layout()
plt.savefig('01_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

nov = monthly.loc['2017-11']
prev = monthly.loc['2017-10']
print('Peak revenue month:', peak, f"(R$ {monthly.loc[peak, 'revenue']:,.0f})")
print(f"Nov 2017 vs Oct 2017: revenue {nov['revenue']/prev['revenue']-1:+.0%}, orders {nov['orders']/prev['orders']-1:+.0%}")
print(f"Jan 2018 vs Jan 2017: revenue x{monthly.loc['2018-01','revenue']/monthly.loc['2017-01','revenue']:.1f}")

## 6. Top product categories

In [ ]:
cat = (it.groupby('category')
         .agg(revenue=('price', 'sum'), orders=('order_id', 'nunique'), avg_price=('price', 'mean'))
         .sort_values('revenue', ascending=False))
cat['share'] = cat['revenue'] / cat['revenue'].sum()
top = cat.head(10)

plt.figure(figsize=(8, 4.8))
sns.barplot(x=top['revenue'], y=top.index, color='#1f6feb')
plt.gca().xaxis.set_major_formatter(money)
plt.title('Top 10 categories by revenue')
plt.xlabel(''); plt.ylabel('')
plt.savefig('02_top_categories.png', dpi=150, bbox_inches='tight')
plt.show()

print(top.round(3))
print(f"\nTop 10 categories = {top['share'].sum():.1%} of revenue, out of {cat.shape[0]} categories")

## 7. Revenue by customer state

In [ ]:
state = (it.merge(delivered[['order_id', 'customer_id']], on='order_id')
           .merge(customers[['customer_id', 'customer_state']], on='customer_id')
           .groupby('customer_state')
           .agg(revenue=('price', 'sum'), orders=('order_id', 'nunique'))
           .sort_values('revenue', ascending=False))
state['share'] = state['revenue'] / state['revenue'].sum()

plt.figure(figsize=(8, 4.5))
sns.barplot(x=state.head(10).index, y=state.head(10)['revenue'], color='#8250df')
plt.gca().yaxis.set_major_formatter(money)
plt.title('Top 10 states by revenue')
plt.xlabel(''); plt.ylabel('')
plt.savefig('03_revenue_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

print(state.head(5).round(3))
print(f"\nSP + RJ + MG = {state.head(3)['share'].sum():.1%} of revenue")

## 8. Delivery performance

In [ ]:
st = (dl.groupby('customer_state')
        .agg(avg_days=('delivery_days', 'mean'), late_rate=('late', 'mean'), orders=('order_id', 'count'))
        .query('orders > 500')
        .sort_values('avg_days'))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(x=st['avg_days'], y=st.index, color='#bf8700', ax=ax[0])
ax[0].set_title('Average delivery time (days) by state')
sns.barplot(x=st['late_rate'] * 100, y=st.index, color='#cf222e', ax=ax[1])
ax[1].set_title('Late delivery rate (%) by state')
for a in ax:
    a.set_xlabel(''); a.set_ylabel('')
plt.tight_layout()
plt.savefig('04_delivery_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average delivery: {dl['delivery_days'].mean():.1f} days (median {dl['delivery_days'].median():.1f})")
print(f"Late deliveries: {dl['late'].mean():.1%} of delivered orders")
print('Fastest:', st.index[0], f"{st['avg_days'].iloc[0]:.1f} days | Slowest:", st.index[-1], f"{st['avg_days'].iloc[-1]:.1f} days")

## 9. Does delivery affect customer reviews?

In [ ]:
rv = dl.dropna(subset=['review_score']).copy()
rv['status'] = np.where(rv['late'], 'Late', 'On time')
rv['bucket'] = pd.cut(rv['delivery_days'], [0, 7, 14, 21, 30, 500],
                      labels=['0-7 days', '8-14', '15-21', '22-30', '30+'])

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
dist = rv['review_score'].round().value_counts(normalize=True).sort_index() * 100
sns.barplot(x=dist.index.astype(int), y=dist.values, color='#1f6feb', ax=ax[0])
ax[0].set_title('Review score distribution (%)')
ax[0].set_xlabel('Stars'); ax[0].set_ylabel('')

by_status = rv.groupby('status')['review_score'].mean()
sns.barplot(x=by_status.index, y=by_status.values, hue=by_status.index, palette=['#2da44e', '#cf222e'], legend=False, ax=ax[1])
ax[1].set_title('Avg review score: on time vs late')
ax[1].set_xlabel(''); ax[1].set_ylabel('')
ax[1].set_ylim(0, 5)

by_bucket = rv.groupby('bucket', observed=True)['review_score'].mean()
sns.barplot(x=by_bucket.index, y=by_bucket.values, color='#bf8700', ax=ax[2])
ax[2].set_title('Avg review score by delivery time')
ax[2].set_xlabel(''); ax[2].set_ylabel('')
ax[2].set_ylim(0, 5)
plt.tight_layout()
plt.savefig('05_reviews_vs_delivery.png', dpi=150, bbox_inches='tight')
plt.show()

low = rv.assign(low=rv['review_score'] <= 2).groupby('status')['low'].mean()
print(by_status.round(2).to_string())
print()
print('Share of 1-2 star reviews:'); print((low * 100).round(1).to_string())
print()
print(by_bucket.round(2).to_string())

## 10. Payments and repeat customers

In [ ]:
pay = payments.merge(delivered[['order_id']], on='order_id')
pt = pay.groupby('payment_type')['payment_value'].agg(['sum', 'count'])
pt['share_of_value'] = pt['sum'] / pt['sum'].sum()
pt = pt.sort_values('sum', ascending=False)
print(pt.round(3))
cc = pay[pay['payment_type'] == 'credit_card']
print(f"\nAvg credit card installments: {cc['payment_installments'].mean():.1f}")
print(f"Customers with 2+ orders: {repeat_rate:.1%} of {orders_per_cust.shape[0]:,}")

plt.figure(figsize=(5.5, 4))
plt.pie(pt['sum'], labels=pt.index, autopct='%1.0f%%', startangle=90, colors=sns.color_palette('Set2'))
plt.title('Payment value by type')
plt.savefig('06_payment_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Seller concentration

In [ ]:
seller_rev = it.groupby('seller_id')['price'].sum().sort_values(ascending=False)
cum = seller_rev.cumsum() / seller_rev.sum()
x = np.arange(1, len(cum) + 1) / len(cum) * 100
top10_share = seller_rev.head(int(len(seller_rev) * 0.1)).sum() / seller_rev.sum()

plt.figure(figsize=(6, 4.2))
plt.plot(x, cum.values * 100, color='#1f6feb')
plt.axvline(10, color='gray', ls='--'); plt.axhline(top10_share * 100, color='gray', ls='--')
plt.xlabel('% of sellers (ranked by revenue)'); plt.ylabel('% of revenue')
plt.title('Seller revenue concentration')
plt.savefig('07_seller_concentration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'{len(seller_rev):,} sellers; top 10% of sellers earn {top10_share:.1%} of revenue')

## 12. Key insights and recommendations

1. **Strong growth with a Black Friday spike.** Revenue grew sharply through 2017 and flattened in 2018. November 2017 was the peak month.
2. **Revenue is concentrated geographically.** São Paulo (SP) alone is the biggest market, and SP, RJ and MG together bring most of the revenue. Other states are an expansion opportunity if delivery improves.
3. **Late deliveries hurt reviews.** Late orders have a much lower average score and far more 1-2 star reviews. Reducing late deliveries is the clearest way to improve customer satisfaction.
4. **Delivery is slow far from the southeast.** Northern and northeastern states wait longer and see higher late rates, so delivery estimates there should be more realistic.
5. **Very few customers return.** Only a small share of customers order twice, so retention campaigns (email offers, loyalty rewards) are a big opportunity.
6. **Credit cards dominate payments**, usually paid in several installments.
7. **A small group of sellers drives most revenue.** The top 10% of sellers bring about two thirds of sales, so seller support and monitoring matters.

*Note: these are associations, not proof of cause. For example, late orders may also differ in product type or distance.*

## 13. Download outputs
Save the charts and KPI table, then upload them to your GitHub repo.

In [ ]:
for f in ['kpis.csv', '01_monthly_trend.png', '02_top_categories.png', '03_revenue_by_state.png',
          '04_delivery_by_state.png', '05_reviews_vs_delivery.png', '06_payment_types.png',
          '07_seller_concentration.png']:
    files.download(f)